# Gyaradax Validation Suite

Comprehensive validation of the `gyaradax` solver against GKW standard reference tests,
covering both **adiabatic** and **kinetic electron** physics.
All test cases are electrostatic ($\beta = 0$ or negligibly small).

**Adiabatic Electron Tests** (`adiabatic_electrons = True`, 1 kinetic species):
1. `eiv_simple` — Linear CBC eigenvalue ($\hat{s}=0.78$, $q=1.4$, circ)
2. `slab_itg` — Linear slab ITG ($\hat{s}=1.0$, $q=1.0$, slab_periodic)
3. `miller_mb` — Linear Miller geometry ($\hat{s}=1.0$, $q=2.0$, Miller)
4. `sourcetime` — Nonlinear CBC ($\hat{s}=0.78$, $q=1.4$, circ, mode_box)

**Kinetic Electron Tests** (`adiabatic_electrons = False`, 2 kinetic species):
5. `kinetic_elec` — Linear Miller, electron-driven ITG
6. `geom_circ` — Linear circular, symmetric gradients

**Long-running Validation** (pre-computed `validation_outputs_*` / `validation_kinetic_*`):
- Flux traces, spectra, and performance comparisons

In [ ]:
import sys
sys.path.append("..")

from gyaradax.bootstrap import init_jax
init_jax(device=6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gyaradax.plot_utils import (
    plot_flux_trace,
    plot_spectra,
    JAX_COLORS,
    SPECIES_COLORS,
)
from gyaradax.params import load_config
from gyaradax.utils import load_geometry

GKW_TESTS = os.path.join("..", "gkw_ref", "tests", "standard")
os.makedirs("figs", exist_ok=True)

---
# Part I — Adiabatic Electron Tests

GKW reference data for single-species, electrostatic test cases.

## 1. `eiv_simple` — Linear CBC Eigenvalue

Circular geometry, $R/L_T = 6.9$, $R/L_n = 2.2$, $q = 1.4$, $\hat{s} = 0.78$, $\epsilon = 0.19$.

Reference growth rate: $\gamma = 1.81840 \times 10^{-1}$ at $k_\theta\rho_s = 0.5$.

In [ ]:
eiv_dir = os.path.join(GKW_TESTS, "eiv_simple", "reference")

# time.dat: (time, growth_rate, frequency) — single converged snapshot
eiv_time = np.loadtxt(os.path.join(eiv_dir, "time.dat")).reshape(1, -1)
# fluxes.dat: (pflux, eflux, vflux)
eiv_fluxes = np.loadtxt(os.path.join(eiv_dir, "fluxes.dat")).reshape(1, -1)

print("eiv_simple (converged eigenvalue):")
print(f"  time = {eiv_time[0, 0]:.2f}")
print(f"  growth rate = {eiv_time[0, 1]:.5e}  (target: 1.81840e-01)")
print(f"  frequency   = {eiv_time[0, 2]:.5e}")
print(f"  pflux = {eiv_fluxes[0, 0]:.5e}")
print(f"  eflux = {eiv_fluxes[0, 1]:.5e}")
print(f"  vflux = {eiv_fluxes[0, 2]:.5e}")

## 2. `slab_itg` — Linear Slab ITG

Slab periodic geometry, $R/L_T = 9.0$, $R/L_n = 0.25$, $q = 1.0$, $\hat{s} = 1.0$, $\epsilon = 1.0$,
$k_\theta\rho_s = 0.5$, mode_box, $\beta \approx 0$.

Reference: growth rate convergence and flux evolution over 20 time windows.

In [ ]:
slab_dir = os.path.join(GKW_TESTS, "slab_itg", "reference")

# time.dat: (time, growth_rate, frequency) per NAVERAGE window
slab_time = np.loadtxt(os.path.join(slab_dir, "time.dat"))
# fluxes.dat: (pflux, eflux, vflux)
slab_fluxes = np.loadtxt(os.path.join(slab_dir, "fluxes.dat"))

print(
    f"slab_itg: {len(slab_time)} time windows, t=[{slab_time[0,0]:.1f}, {slab_time[-1,0]:.1f}]"
)
print(f"  final growth rate = {slab_time[-1,1]:.5e}")
print(f"  final frequency   = {slab_time[-1,2]:.5e}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].plot(slab_time[:, 0], slab_time[:, 1], "o-", color=JAX_COLORS["blue"], ms=3)
axes[0].set_xlabel(r"time $[v_{th}/R]$")
axes[0].set_ylabel(r"$\gamma$")
axes[0].set_title("Growth Rate")
axes[0].grid(True)

axes[1].plot(slab_time[:, 0], slab_time[:, 2], "o-", color=JAX_COLORS["red"], ms=3)
axes[1].set_xlabel(r"time $[v_{th}/R]$")
axes[1].set_ylabel(r"$\omega$")
axes[1].set_title("Frequency")
axes[1].grid(True)

axes[2].semilogy(
    slab_time[:, 0], slab_fluxes[:, 1], "o-", color=JAX_COLORS["green"], ms=3
)
axes[2].set_xlabel(r"time $[v_{th}/R]$")
axes[2].set_ylabel("eflux")
axes[2].set_title("Heat Flux")
axes[2].grid(True)

fig.suptitle("slab_itg (GKW reference)", fontweight="bold")
fig.tight_layout()
fig.savefig("figs/gkw_ref_slab_itg.pdf")
plt.show()

## 3. `miller_mb` — Linear Miller Geometry

Miller geometry, $R/L_T = 9.0$, $R/L_n = 3.0$, $q = 2.0$, $\hat{s} = 1.0$, $\epsilon = 0.16$,
$k_\theta\rho_s = 0.3$, mode_box ($k_{y,\max} = 0.5$), $\beta = 3 \times 10^{-4}$.

Includes Miller shaping: $\kappa = 1.4$, $\delta = -0.3$, squareness $= 0.2$.

In [ ]:
miller_dir = os.path.join(GKW_TESTS, "miller_mb", "reference")

miller_time = np.loadtxt(os.path.join(miller_dir, "time.dat"))
miller_fluxes = np.loadtxt(os.path.join(miller_dir, "fluxes.dat"))

print(
    f"miller_mb: {len(miller_time)} time windows, t=[{miller_time[0,0]:.1f}, {miller_time[-1,0]:.1f}]"
)
print(f"  final growth rate = {miller_time[-1,1]:.5e}")
print(f"  final frequency   = {miller_time[-1,2]:.5e}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].plot(miller_time[:, 0], miller_time[:, 1], "o-", color=JAX_COLORS["blue"], ms=4)
axes[0].set_xlabel(r"time $[v_{th}/R]$")
axes[0].set_ylabel(r"$\gamma$")
axes[0].set_title("Growth Rate")
axes[0].grid(True)

axes[1].plot(miller_time[:, 0], miller_time[:, 2], "o-", color=JAX_COLORS["red"], ms=4)
axes[1].set_xlabel(r"time $[v_{th}/R]$")
axes[1].set_ylabel(r"$\omega$")
axes[1].set_title("Frequency")
axes[1].grid(True)

axes[2].plot(
    miller_time[:, 0], miller_fluxes[:, 1], "o-", color=JAX_COLORS["green"], ms=4
)
axes[2].set_xlabel(r"time $[v_{th}/R]$")
axes[2].set_ylabel("eflux")
axes[2].set_title("Heat Flux")
axes[2].grid(True)

fig.suptitle("miller_mb (GKW reference)", fontweight="bold")
fig.tight_layout()
fig.savefig("figs/gkw_ref_miller_mb.pdf")
plt.show()

## 4. `sourcetime` — Nonlinear CBC

Nonlinear Cyclone Base Case with source modulation.
Circular geometry, $R/L_T = 5.0$, $R/L_n = 2.2$, $q = 1.4$, $\hat{s} = 0.78$, $\epsilon = 0.19$.
Mode box: $k_{y,\max} = 1.57$, $N_x = 40$, $L_x = 120$.

Reference: time-averaged electrostatic heat and momentum fluxes.

In [ ]:
src_dir = os.path.join(GKW_TESTS, "sourcetime", "reference")

src_eflux = np.loadtxt(os.path.join(src_dir, "eflux_es.dat"))
src_vflux = np.loadtxt(os.path.join(src_dir, "vflux_es.dat"))

print(f"sourcetime reference ({len(src_eflux)} windows):")
print(f"  eflux_es: {src_eflux}")
print(f"  vflux_es: {src_vflux}")
print(f"  mean eflux = {np.mean(src_eflux):.5e}")
print(f"  mean vflux = {np.mean(src_vflux):.5e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

ax1.bar(range(len(src_eflux)), src_eflux, color=JAX_COLORS["cyan"], alpha=0.8)
ax1.axhline(
    np.mean(src_eflux), color="k", ls="--", lw=1, label=f"mean={np.mean(src_eflux):.2e}"
)
ax1.set_xlabel("window")
ax1.set_ylabel(r"$Q_{es}$")
ax1.set_title("Heat Flux")
ax1.legend(frameon=False)
ax1.grid(True, axis="y")

ax2.bar(range(len(src_vflux)), src_vflux, color=JAX_COLORS["purple"], alpha=0.8)
ax2.axhline(
    np.mean(src_vflux), color="k", ls="--", lw=1, label=f"mean={np.mean(src_vflux):.2e}"
)
ax2.set_xlabel("window")
ax2.set_ylabel(r"$\Gamma_{es}$")
ax2.set_title("Momentum Flux")
ax2.legend(frameon=False)
ax2.grid(True, axis="y")

fig.suptitle("sourcetime (GKW reference)", fontweight="bold")
fig.tight_layout()
fig.savefig("figs/gkw_ref_sourcetime.pdf")
plt.show()

## 5. Adiabatic Validation Runs

Compare pre-computed gyaradax nonlinear runs (`validation_outputs_*`) against their
GKW reference data. Flux traces and spectra side-by-side.

In [ ]:
adiabatic_dirs = sorted(
    [d for d in os.listdir("..") if d.startswith("validation_outputs_")]
)
print(f"found {len(adiabatic_dirs)} adiabatic validation directories: {adiabatic_dirs}")

In [ ]:
for out_dir in adiabatic_dirs:
    path = os.path.join("..", out_dir)
    flux_path = os.path.join(path, "fluxes.npz")
    growth_path = os.path.join(path, "growth.npz")

    if not os.path.exists(flux_path) or not os.path.exists(growth_path):
        print(f"skipping {out_dir}: missing .npz files")
        continue

    sim_flux = np.load(flux_path)["fluxes"]
    sim_time = np.load(growth_path)["time"]

    config_name = out_dir.replace("validation_outputs_", "")
    config_path = os.path.join("..", "configs", f"{config_name}.yaml")

    ref_time, ref_fluxes = None, None
    kx, ky = None, None
    ref_dir = None
    if os.path.exists(config_path):
        cfg = load_config(config_path)
        ref_dir = cfg.run.data_dir
        ref_time_path = os.path.join(ref_dir, "time.dat")
        ref_flux_path = os.path.join(ref_dir, "fluxes.dat")
        if os.path.exists(ref_time_path) and os.path.exists(ref_flux_path):
            ref_time = np.loadtxt(ref_time_path)
            ref_fluxes = np.loadtxt(ref_flux_path).T

        geom = load_geometry(ref_dir)
        kx = np.asarray(geom["kxrh"])
        ky = np.asarray(geom["krho"])

    fig = plot_flux_trace(
        sim_time,
        sim_flux.T[[1, 2]],
        ref_time=ref_time,
        ref_fluxes=ref_fluxes[[1, 2]] if ref_fluxes is not None else None,
        labels=["Heat", "Momentum"],
        title=f"{config_name}",
    )
    fig.savefig(f"figs/fluxes_{config_name}.pdf")
    plt.show()

    # spectra
    kx_spec_path = os.path.join(path, "kxspec.npz")
    ky_spec_path = os.path.join(path, "kyspec.npz")
    if (
        os.path.exists(kx_spec_path)
        and os.path.exists(ky_spec_path)
        and kx is not None
        and ref_dir is not None
    ):
        avg_count = 300
        kx_spec_avg = np.mean(np.load(kx_spec_path)["kx_spec"][-avg_count:], axis=0)
        ky_spec_avg = np.mean(np.load(ky_spec_path)["ky_spec"][-avg_count:], axis=0)

        ref_kx_spec, ref_ky_spec = None, None
        ref_kx_path = os.path.join(ref_dir, "kxspec")
        ref_ky_path = os.path.join(ref_dir, "kyspec")
        if os.path.exists(ref_kx_path) and os.path.exists(ref_ky_path):
            ref_kx_spec = np.mean(np.loadtxt(ref_kx_path)[-avg_count:], axis=0)
            ref_ky_spec = np.mean(np.loadtxt(ref_ky_path)[-avg_count:], axis=0)

        fig_spec = plot_spectra(
            kx=kx,
            ky=ky,
            kx_spec=kx_spec_avg,
            ky_spec=ky_spec_avg,
            ref_kx_spec=ref_kx_spec,
            ref_ky_spec=ref_ky_spec,
            title=f"{config_name}",
        )
        fig_spec.savefig(f"figs/spectra_{config_name}.pdf")
        plt.show()

---
# Part II — Kinetic Electron Tests

## 6. GKW Standard Test Reference Data

GKW reference flux traces for the kinetic electron test cases.
These are the ground-truth targets that gyaradax should match.

In [ ]:
KINETIC_CASES = {
    "kinetic_elec": {
        "title": "kinetic_elec (Miller)",
        "desc": (
            r"$q=1.4$, $\hat{s}=0.522$, $\epsilon=0.173$, $k_\theta\rho_s=0.424$"
            "\nIon: $R/L_T=0$, $R/L_n=1.05$ | "
            "Elec: $R/L_T=9.15$, $R/L_n=1.05$"
        ),
    },
    "geom_circ": {
        "title": "geom_circ (Circular)",
        "desc": (
            r"$q=2.0$, $\hat{s}=1.0$, $\epsilon=0.16$, $k_\theta\rho_s=0.2$"
            "\nBoth species: $R/L_T=9.0$, $R/L_n=3.0$"
        ),
    },
}

for case_name, info in KINETIC_CASES.items():
    ref_dir = os.path.join(GKW_TESTS, case_name, "reference")
    time_path = os.path.join(ref_dir, "time.dat")
    flux_path = os.path.join(ref_dir, "fluxes.dat")

    if not os.path.exists(time_path) or not os.path.exists(flux_path):
        print(f"skipping {case_name}: reference data not found")
        continue

    ref_time = np.loadtxt(time_path)
    ref_fluxes = np.loadtxt(flux_path)  # (ntime, 6): pf_i, ef_i, vf_i, pf_e, ef_e, vf_e

    # time.dat may have 3 columns (time, growth, freq) — use col 0
    if ref_time.ndim == 2:
        t = ref_time[:, 0]
    else:
        t = ref_time

    print(f"\n{'='*60}")
    print(f"{info['title']}")
    print(f"  steps: {len(t)}, t=[{t[0]:.4f}, {t[-1]:.4f}]")
    print(f"  fluxes shape: {ref_fluxes.shape}")

    fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True)

    flux_labels = ["pflux", "eflux"]
    for i, label in enumerate(flux_labels):
        axes[i, 0].plot(t, ref_fluxes[:, i], color=SPECIES_COLORS[0], lw=1.2)
        axes[i, 0].set_ylabel(label)
        axes[i, 0].grid(True, axis="y")
        if i == 0:
            axes[i, 0].set_title("ion")

        axes[i, 1].plot(t, ref_fluxes[:, i + 3], color=SPECIES_COLORS[1], lw=1.2)
        axes[i, 1].grid(True, axis="y")
        if i == 0:
            axes[i, 1].set_title("electron")

    for ax in axes[-1]:
        ax.set_xlabel(r"time $[v_{th}/R]$")

    fig.suptitle(f"GKW Reference: {info['title']}", fontweight="bold")
    fig.tight_layout()
    fig.savefig(f"figs/gkw_ref_{case_name}.pdf")
    plt.show()

## 7. Kinetic Electron Validation: Flux Traces

Compare pre-computed gyaradax kinetic electron simulations against GKW reference trajectories.
Per-species ion and electron heat fluxes shown side-by-side.

In [ ]:
KINETIC_REF_DIR = "/restricteddata/ukaea/gyrokinetics/raw/kinetic_electrons"

kinetic_dirs = sorted(
    [d for d in os.listdir("..") if d.startswith("validation_kinetic_")]
)
print(f"found {len(kinetic_dirs)} kinetic validation directories: {kinetic_dirs}")

In [ ]:
for out_dir in kinetic_dirs:
    path = os.path.join("..", out_dir)
    results_path = os.path.join(path, "results.npz")
    if not os.path.exists(results_path):
        print(f"skipping {out_dir}: no results.npz")
        continue

    data = np.load(results_path)
    sim_times = data["sim_times"]
    sim_eflux_ion = data["sim_eflux_ion"]
    sim_eflux_elec = data["sim_eflux_elec"]

    case_name = out_dir.replace("validation_kinetic_", "")
    ref_dir = os.path.join(KINETIC_REF_DIR, case_name)

    if not os.path.exists(os.path.join(ref_dir, "time.dat")):
        print(f"skipping {case_name}: no reference data at {ref_dir}")
        continue

    ref_time = np.loadtxt(os.path.join(ref_dir, "time.dat"))
    ref_fluxes = np.loadtxt(os.path.join(ref_dir, "fluxes.dat"))

    fig, (ax_ion, ax_elec) = plt.subplots(1, 2, figsize=(12, 3), sharex=True)

    ax_ion.plot(
        sim_times, sim_eflux_ion, color=SPECIES_COLORS[0], lw=1.5, label="gyaradax"
    )
    ax_ion.plot(ref_time, ref_fluxes[:, 1], "k--", alpha=0.7, lw=1.2, label="GKW")
    ax_ion.set_ylabel("eflux")
    ax_ion.set_title("ion")
    ax_ion.legend(frameon=False)
    ax_ion.grid(True, axis="y")

    ax_elec.plot(
        sim_times, sim_eflux_elec, color=SPECIES_COLORS[1], lw=1.5, label="gyaradax"
    )
    ax_elec.plot(ref_time, ref_fluxes[:, 4], "k--", alpha=0.7, lw=1.2, label="GKW")
    ax_elec.set_title("electron")
    ax_elec.legend(frameon=False)
    ax_elec.grid(True, axis="y")

    for ax in (ax_ion, ax_elec):
        ax.set_xlabel(r"time $[v_{th}/R]$")

    fig.suptitle(case_name, fontweight="bold")
    fig.tight_layout()
    fig.savefig(f"figs/kinetic_fluxes_{case_name}.pdf")
    plt.show()

    # time-averaged comparison over last 25%
    avg_start = max(0, len(sim_eflux_ion) - len(sim_eflux_ion) // 4)
    sim_avg_i = np.mean(sim_eflux_ion[avg_start:])
    sim_avg_e = np.mean(sim_eflux_elec[avg_start:])

    ref_eflux_ion, ref_eflux_elec = [], []
    for t in sim_times[avg_start:]:
        idx = np.argmin(np.abs(ref_time - t))
        ref_eflux_ion.append(ref_fluxes[idx, 1])
        ref_eflux_elec.append(ref_fluxes[idx, 4])

    ref_avg_i = np.mean(ref_eflux_ion)
    ref_avg_e = np.mean(ref_eflux_elec)

    print(
        f"\n{case_name} time-averaged eflux (last {len(sim_eflux_ion) - avg_start} blocks):"
    )
    print(
        f"  ion:      sim={sim_avg_i:.4e}  ref={ref_avg_i:.4e}  "
        f"rel_err={abs(sim_avg_i - ref_avg_i) / max(abs(ref_avg_i), 1e-15):.2e}"
    )
    print(
        f"  electron: sim={sim_avg_e:.4e}  ref={ref_avg_e:.4e}  "
        f"rel_err={abs(sim_avg_e - ref_avg_e) / max(abs(ref_avg_e), 1e-15):.2e}"
    )

## 8. Kinetic Electron Spectra

For nonlinear kinetic validation runs that saved spectral data, compare
$k_x$ and $k_y$ power spectra against GKW reference.

In [ ]:
for out_dir in kinetic_dirs:
    path = os.path.join("..", out_dir)
    kx_spec_path = os.path.join(path, "kxspec.npz")
    ky_spec_path = os.path.join(path, "kyspec.npz")

    if not os.path.exists(kx_spec_path) or not os.path.exists(ky_spec_path):
        print(f"skipping {out_dir}: no spectral data")
        continue

    case_name = out_dir.replace("validation_kinetic_", "")
    ref_dir = os.path.join(KINETIC_REF_DIR, case_name)

    geom = load_geometry(ref_dir)
    kx = np.asarray(geom["kxrh"])
    ky = np.asarray(geom["krho"])

    avg_count = 300
    kx_spec_avg = np.mean(np.load(kx_spec_path)["kx_spec"][-avg_count:], axis=0)
    ky_spec_avg = np.mean(np.load(ky_spec_path)["ky_spec"][-avg_count:], axis=0)

    ref_kx_spec, ref_ky_spec = None, None
    ref_kx_path = os.path.join(ref_dir, "kxspec")
    ref_ky_path = os.path.join(ref_dir, "kyspec")
    if os.path.exists(ref_kx_path) and os.path.exists(ref_ky_path):
        ref_kx_spec = np.mean(np.loadtxt(ref_kx_path)[-avg_count:], axis=0)
        ref_ky_spec = np.mean(np.loadtxt(ref_ky_path)[-avg_count:], axis=0)

    fig = plot_spectra(
        kx=kx,
        ky=ky,
        kx_spec=kx_spec_avg,
        ky_spec=ky_spec_avg,
        ref_kx_spec=ref_kx_spec,
        ref_ky_spec=ref_ky_spec,
        title=f"{case_name} (kinetic)",
    )
    fig.savefig(f"figs/kinetic_spectra_{case_name}.pdf")
    plt.show()

---
# Part III — Performance

## 9. Solver Throughput

Steps per second across all validation runs (adiabatic + kinetic).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
has_data = False

for out_dir in adiabatic_dirs + kinetic_dirs:
    perf_path = os.path.join("..", out_dir, "performance.npz")
    if os.path.exists(perf_path):
        perf = np.load(perf_path)
        if "steps_per_sec" in perf:
            label = out_dir.replace("validation_outputs_", "").replace(
                "validation_kinetic_", "kin:"
            )
            ax.plot(perf["steps_per_sec"], "o-", ms=2, label=label)
            has_data = True

if has_data:
    ax.set_xlabel("block")
    ax.set_ylabel("steps/s")
    ax.set_title("solver throughput")
    ax.legend(frameon=False)
    ax.grid(True, linestyle=":", alpha=0.5)
else:
    ax.text(
        0.5,
        0.5,
        "no performance data found",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )
plt.show()